# CEH GEAR-Hourly: Zarr via Python + Xarray

**Launch this notebook:**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/YOUR_REPO/blob/main/gear_zarr_python.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/YOUR_ORG/YOUR_REPO/HEAD?labpath=gear_zarr_python.ipynb)
[![Open in DataLabs](https://img.shields.io/badge/Open%20in-NERC%20DataLabs-blue?logo=jupyter)](https://datalabs.nerc.ac.uk)

> Replace `YOUR_ORG/YOUR_REPO` with your GitHub path once published.

---

## What this notebook does

This notebook shows how to explore the **CEH GEAR-Hourly** gridded rainfall dataset for Great Britain — stored in Zarr format on JASMIN object storage — using **xarray**, which handles the Zarr format natively.

**We will:**
1. Open the remote Zarr store and inspect its metadata
2. Extract and plot a time series at a single location
3. Plot a map of rainfall at a single time step

**Zarr store:**
```
https://fdri-o.s3-ext.jc.rl.ac.uk/gearhrly/gearhrly_15day_100km_chunks.zarr
```

**Libraries used:** `xarray`, `matplotlib`  
Both are pre-installed on Colab, Binder, and most scientific Python environments.

| Runtime | Notes |
|---------|-------|
| Google Colab | ✅ Works out of the box — no installs needed |
| Binder | ✅ Works out of the box |
| NERC DataLabs | ✅ Best performance — data is co-located on JASMIN |
| JupyterLite | ✅ Should work with Pyodide kernel |
| Local Jupyter | ✅ Run `pip install xarray matplotlib` if needed |
| Quarto | ✅ Use `jupyter` engine with a Python kernel |

---

## 0. Install and import libraries

On most platforms `xarray` and `matplotlib` are already available.  
The install cell below is harmless to run even if they are already present.

In [ ]:
# Install if needed (safe to run even if already installed)
%pip install -q xarray matplotlib

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

print(f"xarray  version: {xr.__version__}")

---
## 1. Open the Zarr store

`xr.open_zarr()` connects to the store over HTTPS without downloading any data yet.  
Data is only fetched when you actually access specific values.

In [ ]:
store_url = "https://fdri-o.s3-ext.jc.rl.ac.uk/gearhrly/gearhrly_15day_100km_chunks.zarr"

ds = xr.open_zarr(store_url)

# Display the dataset — shows dimensions, variables, coordinates and attributes
ds

---
## 2. Explore the metadata

### 2a. Dataset-level (global) attributes

In [ ]:
print("=== Global attributes ===")
for name, value in ds.attrs.items():
    print(f"  {name:<25} : {value}")

### 2b. Dimensions and coordinates

In [ ]:
print("=== Dimensions ===")
for dim, size in ds.dims.items():
    print(f"  {dim:<15} : {size} values")

print("\n=== Coordinates ===")
for name, coord in ds.coords.items():
    print(f"  {name:<15} : {coord.values.min():.4f} to {coord.values.max():.4f}  [{coord.attrs.get('units', '')}]")

### 2c. Data variables

In [ ]:
print("=== Data variables ===")
for name, var in ds.data_vars.items():
    print(f"\n  Variable : {name}")
    print(f"  Shape    : {var.shape}")
    print(f"  Dims     : {var.dims}")
    for attr, val in var.attrs.items():
        print(f"  {attr:<20} : {val}")

---
## 3. Time series at a single location

xarray's `.sel(method='nearest')` finds the closest grid point to any longitude/latitude you provide — no manual index-searching needed.

In [ ]:
# Choose a location — Edinburgh, Scotland
# Adjust to any point within the dataset's extent
target_lon = -3.19
target_lat = 55.95

# The variable name — adjust if your store uses a different name
var_name = "rainfall_amount"

# Select the nearest grid cell
# The coordinate names depend on the store — common names: longitude/latitude, lon/lat, x/y
point = ds[var_name].sel(longitude=target_lon, latitude=target_lat, method="nearest")

# Load the time series into memory
ts = point.load()

actual_lon = float(point.longitude)
actual_lat = float(point.latitude)
print(f"Target  : lon={target_lon}, lat={target_lat}")
print(f"Nearest : lon={actual_lon:.3f}, lat={actual_lat:.3f}")
print(f"Time steps: {len(ts)}")

In [ ]:
units = ds[var_name].attrs.get("units", "")
long_name = ds[var_name].attrs.get("long_name", var_name)

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(ts.time, ts.values, color="#1d6fa4", linewidth=0.8, alpha=0.85)

ax.set_title(f"GEAR-Hourly: {long_name}", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel(f"{long_name} ({units})")
ax.text(0.01, 0.97, f"lon={actual_lon:.3f}°, lat={actual_lat:.3f}°",
        transform=ax.transAxes, va="top", fontsize=9, color="grey")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---
## 4. Map at a single time step

Select one time step and load the full spatial grid. xarray's `.plot()` creates a quick map from any 2-D DataArray.

In [ ]:
# Select the first time step
# You can use isel(time=N) for index-based selection,
# or sel(time="YYYY-MM-DD") for date-based selection
t_index = 0
slice_2d = ds[var_name].isel(time=t_index).load()

time_label = str(slice_2d.time.values)[:16]   # trim to minute precision
print(f"Time step : {time_label}")
print(f"Grid shape: {slice_2d.shape}")
print(f"Value range: {float(slice_2d.min()):.3f} to {float(slice_2d.max()):.3f} {units}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))

# xarray's built-in .plot() handles axis labels and colorbars automatically
slice_2d.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"{long_name} ({units})", "shrink": 0.7}
)

ax.set_title(f"GEAR-Hourly: {long_name}\n{time_label}", fontsize=12, fontweight="bold")
ax.set_aspect("equal")   # preserve grid cell shape
fig.tight_layout()
plt.show()

---
## 5. Accumulated rainfall over a time window

Sum the first 24 time steps (one day of hourly data) over the full grid.  
xarray handles the aggregation in one line — only the needed chunks are fetched.

In [ ]:
n_steps = min(24, len(ds.time))

total_24h = ds[var_name].isel(time=slice(0, n_steps)).sum(dim="time").load()

t_start = str(ds.time.values[0])[:16]
t_end   = str(ds.time.values[n_steps - 1])[:16]

fig, ax = plt.subplots(figsize=(6, 8))

total_24h.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"{n_steps}-hour total ({units})", "shrink": 0.7}
)

ax.set_title(
    f"GEAR-Hourly: {n_steps}-hour accumulated rainfall\n{t_start} to {t_end}",
    fontsize=12, fontweight="bold"
)
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

---
## Tips

| Task | Code |
|------|------|
| Select by date | `ds[var].sel(time="2015-01-01T06:00")` |
| Select a date range | `ds[var].sel(time=slice("2015-01-01", "2015-01-31"))` |
| Monthly mean | `ds[var].resample(time="1ME").mean()` |
| Spatial mean over GB | `ds[var].mean(dim=["latitude", "longitude"])` |
| Save to NetCDF | `ds.to_netcdf("output.nc")` |
| List coordinate names | `list(ds.coords)` |

**Data citation:** CEH-GEAR, UKCEH. https://doi.org/10.5285/dbf13dd5-90cd-457a-a986-f2f9dd97e93c